# 12. IVF — k-means 로 나누고, 가까운 군집만 뒤진다

슬라이드 「IVF 원리」 · 「k-means 로 군집 만들기」 · 「군집화 비용과 운영」 · 「IVF 실제 구동」
1. k-means: 초기 중심 → **할당** → **갱신** → 수렴할 때까지 반복
2. 각 군집의 멤버 목록 = **역색인(Inverted File)**
3. 질의는 중심 `nlist` 개와 비교 → 가까운 `nprobe` 개 군집의 멤버만 정밀 비교

| 셀 | 하는 일 |
|---|---|
| 1 | k-means 구현 |
| 2 | 슬라이드 예제 1·2·3·10·11·12 |
| 3 | 2차원 k-means — 할당과 갱신 |
| 4 | 초기값 — 무작위 vs k-means++ |
| 5 | IVF 검색과 경계 문제 |
| 6 | nlist · nprobe — recall 과 비용 |
| 7 | 학습 비용과 표본 학습 |

**Run → Run All Cells** 로 한 번에 실행하거나, 셀을 하나씩 **Shift + Enter**

### 셀 1 · k-means 구현

In [ ]:
# 셀 1 · k-means 구현
from labutil import *
from matplotlib.colors import ListedColormap

PALETTE = ["#FF7043", "#009CA6", "#3D484A", "#F6BD60", "#84A59D", "#6D597A", "#E56B6F", "#355070",
           "#B56576", "#90BE6D", "#577590", "#F28482", "#43AA8B", "#F9844A", "#277DA1", "#9C89B8"]


def sqdist(A, B):
    """A(n,d) 와 B(k,d) 의 모든 쌍 거리² (n,k)."""
    return (A ** 2).sum(1)[:, None] - 2 * A @ B.T + (B ** 2).sum(1)[None, :]


def kmeans(X, k, iters=50, init="random", seed=0, history=False, stop=True):
    """반환: 중심, 라벨, 반복 횟수, (history=True 면 반복마다 중심 기록)"""
    rng = np.random.default_rng(seed)
    if init == "random":                          # ① 무작위로 k 개를 고른다
        C = X[rng.choice(len(X), k, replace=False)].copy()
    else:                                         # k-means++ : 이미 고른 중심에서 먼 점일수록 잘 뽑힌다
        C = [X[rng.integers(len(X))]]
        d = np.full(len(X), np.inf)
        for _ in range(k - 1):
            d = np.minimum(d, np.maximum(sqdist(X, C[-1][None])[:, 0], 0))   # 가장 가까운 중심까지 거리²
            C.append(X[rng.choice(len(X), p=d / d.sum())])
        C = np.array(C)
    hist = [C.copy()]
    for it in range(1, iters + 1):
        lab = sqdist(X, C).argmin(1)              # ② 할당: 가장 가까운 중심
        newC = np.array([X[lab == j].mean(0) if (lab == j).any() else C[j] for j in range(k)])  # ③ 갱신: 평균
        moved = np.abs(newC - C).max()
        C = newC
        hist.append(C.copy())
        if stop and moved < 1e-6:                 # ④ 중심이 안 움직이면 수렴
            break
    lab = sqdist(X, C).argmin(1)
    return (C, lab, it, hist) if history else (C, lab, it)


def inertia(X, C, lab):
    return float(((X - C[lab]) ** 2).sum())

## 1. 슬라이드 예제: 1, 2, 3, 10, 11, 12

### 셀 2 · 슬라이드 예제 1·2·3·10·11·12

In [ ]:
# 셀 2 · 슬라이드 예제 1·2·3·10·11·12
x = np.array([[1], [2], [3], [10], [11], [12]], dtype=float)
fig, axes = plt.subplots(2, 1, figsize=(9, 3.2), sharex=True)
for ax, init, title in [(axes[0], np.array([[2.0], [11.0]]), "중심 2·11 로 시작 (슬라이드)"),
                        (axes[1], np.array([[1.0], [2.0]]), "나쁜 시작: 중심 1·2")]:
    C = init.copy()
    for it in range(5):
        lab = sqdist(x, C).argmin(1)
        ax.scatter(C[:, 0], [it] * len(C), marker="X", s=120, color="#FF7043", zorder=3)
        newC = np.array([x[lab == j].mean(0) for j in range(2)])
        if np.allclose(newC, C):
            break
        C = newC
    ax.scatter(x[:, 0], [-0.8] * 6, s=80, color="#3D484A")
    ax.set_yticks(range(it + 1)); ax.set_ylabel("반복"); ax.set_title(f"{title} → {it}번째에 수렴, 중심 {C.ravel()}")
    ax.invert_yaxis()
plt.tight_layout(); plt.show()

**관찰:** 나쁘게 시작해도 이 예제는 몇 번 만에 같은 답으로 온다. 2차원·다군집에서는 항상 그렇지는 않다 → 3절.

## 2. 2차원 k-means — 할당과 갱신을 눈으로

### 셀 3 · 2차원 k-means — 할당과 갱신

In [ ]:
# 셀 3 · 2차원 k-means — 할당과 갱신
X, true_lab = make_clusters(1500, 2, n_clusters=6, spread=1.0, center_scale=5, seed=11)
C, lab, n_it, hist = kmeans(X, 6, seed=3, history=True)

gx, gy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                     np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300))
grid = np.c_[gx.ravel(), gy.ravel()]
cmap = ListedColormap(PALETTE[:6])

show = [0, 1, 2, len(hist) - 1]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, i in zip(axes, show):
    Ci = hist[i]
    region = sqdist(grid, Ci).argmin(1).reshape(gx.shape)
    ax.contourf(gx, gy, region, levels=np.arange(7) - 0.5, cmap=cmap, alpha=0.18)
    ax.scatter(*X.T, s=4, c=sqdist(X, Ci).argmin(1), cmap=cmap)
    ax.scatter(*Ci.T, marker="X", s=180, color="black")
    if i > 0:
        for a, b in zip(hist[i - 1], Ci):
            ax.annotate("", b, a, arrowprops=dict(arrowstyle="->", lw=1.5))
    ax.set_title(f"반복 {i}" + (" (수렴)" if i == show[-1] else "")); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

inert = [inertia(X, Ci, sqdist(X, Ci).argmin(1)) for Ci in hist]
plt.figure(figsize=(6, 2.6)); plt.plot(inert, "-o"); plt.xlabel("반복"); plt.ylabel("군집 내 거리² 합")
plt.title(f"{n_it}번 만에 수렴 — 반복마다 줄어들기만 한다"); plt.show()

**관찰:** 배경 색은 "이 위치의 질의는 어느 군집으로 가는가" = **보로노이 영역**.
화살표는 ③ 갱신에서 중심이 멤버의 평균으로 이동하는 모습이다.

## 3. 초기값 의존 — 무작위 vs k-means++
슬라이드: "초기값 의존(지역최적) → k-means++ · 재시작으로 완화"

### 셀 4 · 초기값 — 무작위 vs k-means++

In [ ]:
# 셀 4 · 초기값 — 무작위 vs k-means++
X3, _ = make_clusters(3000, 2, n_clusters=15, spread=0.7, center_scale=8, seed=2)
res = {"random": [], "k-means++": []}
for s in range(20):
    for init in res:
        C3, l3, _ = kmeans(X3, 15, init="random" if init == "random" else "pp", seed=s)
        res[init].append((inertia(X3, C3, l3), C3, l3))

fig = plt.figure(figsize=(15, 4.2))
ax = fig.add_subplot(1, 3, 1)
ax.boxplot([[r[0] for r in v] for v in res.values()], tick_labels=list(res.keys()))
for i, v in enumerate(res.values(), 1):
    ax.scatter(np.full(20, i) + np.random.default_rng(0).uniform(-0.1, 0.1, 20), [r[0] for r in v], s=10, alpha=0.6)
ax.set_title("시드 20개의 군집 내 거리² 합 (낮을수록 좋음)")
for j, (name, pick) in enumerate([("random 최악", max(res["random"], key=lambda r: r[0])),
                                  ("k-means++ 최악", max(res["k-means++"], key=lambda r: r[0]))], 2):
    ax = fig.add_subplot(1, 3, j)
    ax.scatter(*X3.T, s=3, c=pick[2], cmap=ListedColormap(PALETTE[:15]))
    ax.scatter(*pick[1].T, marker="X", s=90, color="black")
    ax.set_title(f"{name}: {pick[0]:,.0f}"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

**관찰:** 무작위 시작은 한 군집에 중심 두 개가 몰리고 다른 두 군집을 중심 하나가 떠안는 경우가 생긴다.
k-means++ 는 서로 먼 점을 중심으로 고르므로 이런 실패가 훨씬 드물다.

## 4. IVF 검색과 경계 문제
16개 군집(nlist=16)으로 역색인을 만들고, **군집 경계 근처**의 질의를 던져 본다.

### 셀 5 · IVF 검색과 경계 문제

In [ ]:
# 셀 5 · IVF 검색과 경계 문제
Xi, _ = make_clusters(3000, 2, n_clusters=10, spread=1.4, center_scale=5, seed=4)
Ci, li, _ = kmeans(Xi, 16, init="pp", seed=0)
lists = {j: np.where(li == j)[0] for j in range(16)}          # 역색인: 군집 번호 → 멤버 id


def ivf_search(qv, C, lists, X, k=1, nprobe=1):
    near = sqdist(qv[None], C)[0].argsort()[:nprobe]           # 중심 nlist 개와 비교 → 가까운 nprobe 개
    cand = np.concatenate([lists[j] for j in near])
    d = sqdist(qv[None], X[cand])[0]
    return cand[d.argsort()[:k]], near, len(C) + len(cand)     # (결과, 뒤진 군집, 거리 계산 수)


# 경계 질의 찾기: nprobe=1 로는 틀리고 nprobe=2 로는 맞는 질의
rng = np.random.default_rng(5)
for _ in range(5000):
    qv = Xi[rng.integers(len(Xi))] + rng.normal(0, 0.3, 2).astype(np.float32)
    t = brute_knn(Xi, qv, 1)[0, 0]
    if ivf_search(qv, Ci, lists, Xi, nprobe=1)[0][0] != t and ivf_search(qv, Ci, lists, Xi, nprobe=2)[0][0] == t:
        break

gx, gy = np.meshgrid(np.linspace(qv[0] - 3, qv[0] + 3, 300), np.linspace(qv[1] - 3, qv[1] + 3, 300))
region = sqdist(np.c_[gx.ravel(), gy.ravel()], Ci).argmin(1).reshape(gx.shape)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.4))
for ax, npb in zip(axes, [1, 2]):
    found, near, cost = ivf_search(qv, Ci, lists, Xi, nprobe=npb)
    ax.contourf(gx, gy, np.isin(region, near), levels=[-0.5, 0.5, 1.5], colors=["white", "#FFE0D6"])
    ax.contour(gx, gy, region, levels=np.arange(17) - 0.5, colors="#9aa5a8", linewidths=0.8)
    m = (np.abs(Xi - qv) < 3).all(1)
    ax.scatter(*Xi[m].T, s=8, color="#3D484A")
    ax.scatter(*qv, marker="*", s=300, color="#009CA6", zorder=4, label="질의")
    ax.scatter(*Xi[t], s=200, facecolors="none", edgecolors="#009CA6", lw=2.5, zorder=4, label="진짜 최근접")
    ax.scatter(*Xi[found[0]], s=80, marker="s", facecolors="none", edgecolors="#FF7043", lw=2, zorder=4, label="IVF 결과")
    ax.set_xlim(qv[0] - 3, qv[0] + 3); ax.set_ylim(qv[1] - 3, qv[1] + 3)
    ax.set_title(f"nprobe={npb}: 칠해진 군집만 뒤짐 · 거리계산 {cost}회 · {'정답' if found[0] == t else '놓침'}")
    ax.legend(loc="lower left", fontsize=8); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

**관찰:** 질의는 A 군집 영역에 있지만 진짜 최근접은 경계 너머 B 군집의 멤버다.
`nprobe=1` 은 A 만 뒤지므로 놓치고, `nprobe=2` 로 옆 군집까지 넣으면 찾는다 (슬라이드 "경계 문제").

## 5. nlist · nprobe — recall 과 비용 (32차원 2만 개)
슬라이드 권장: `nlist ≈ √N`. pgvector 문서는 100만 행 이하에서 `rows/1000` 을 권한다.

### 셀 6 · nlist · nprobe — recall 과 비용

In [ ]:
# 셀 6 · nlist · nprobe — recall 과 비용
N = 20_000
# 군집 중심 간격(center_scale)을 잡음 수준으로 줄여 군집끼리 겹치게 만든다 (실제 임베딩에 가깝게)
Xb, _ = make_clusters(N + 300, 32, n_clusters=60, center_scale=0.8, seed=8)
Xb, Qb = Xb[:N], Xb[N:]
truth = brute_knn(Xb, Qb, 10)

curves = {}
for nlist in [20, 141, 600]:                                  # rows/1000 · √N · 4√N
    C, lab, _ = kmeans(Xb, nlist, init="pp", iters=25, seed=0)
    L = {j: np.where(lab == j)[0] for j in range(nlist)}
    pts = []
    for npb in [1, 2, 4, 8, 16, 32, 64]:
        if npb > nlist:
            continue
        out = [ivf_search(qq, C, L, Xb, k=10, nprobe=npb) for qq in Qb]
        pts.append((npb, recall([o[0] for o in out], truth), np.mean([o[2] for o in out])))
    curves[nlist] = (pts, np.bincount(lab, minlength=nlist))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
for nlist, (pts, _) in curves.items():
    npb, r, c = zip(*pts)
    axes[0].plot(c, r, "-o", label=f"nlist={nlist}")
    for p_, x_, y_ in pts:
        axes[0].annotate(str(p_), (y_, x_), textcoords="offset points", xytext=(3, -10), fontsize=7)
axes[0].set_xscale("log"); axes[0].set_xlabel("질의당 거리 계산 (중심 + 멤버)   ※ 점 옆 숫자 = nprobe")
axes[0].set_ylabel("recall@10"); axes[0].axvline(N, ls="--", color="gray"); axes[0].text(N, 0.3, " 전수 비교", color="gray")
axes[0].legend(); axes[0].set_title("nprobe ↑ → recall ↑ · 비용 ↑")
sizes = np.sort(curves[141][1])[::-1]
axes[1].bar(range(len(sizes)), sizes, color="#009CA6", width=1.0)
axes[1].axhline(sizes.mean(), color="#FF7043", ls="--", label=f"평균 {sizes.mean():.0f}")
axes[1].set_xlabel("군집 (크기 순)"); axes[1].set_ylabel("멤버 수"); axes[1].legend()
axes[1].set_title(f"nlist=141 군집 크기: 최대 {sizes.max()} · 최소 {sizes.min()}")
plt.tight_layout(); plt.show()

**관찰:**
- 같은 `nprobe` 라도 `nlist` 가 크면 군집이 작아 뒤지는 양이 적고, 경계가 많아져 recall 은 낮다
- recall 0.95 를 얻는 **비용**으로 비교하면 √N 근처가 무난하다. `nlist` 가 너무 작으면 한 군집이 너무 크다
- 군집 크기는 고르지 않다 → 큰 리스트를 뒤지는 질의는 느리다 (슬라이드 "검색 성능 들쭉날쭉")

## 6. 학습 비용 N·k·d·T 와 표본 학습
슬라이드: 큰 데이터는 **표본으로 중심만 학습**하고 나머지는 가장 가까운 중심에 배정만 한다.

### 셀 7 · 학습 비용과 표본 학습

In [ ]:
# 셀 7 · 학습 비용과 표본 학습
Xc, _ = make_clusters(40_000, 32, n_clusters=60, center_scale=0.8, seed=9)
rows = []
for n in [10_000, 20_000, 40_000]:
    for k in [50, 100, 200]:
        best = np.inf
        for _ in range(3):                                    # 3번 재서 가장 빠른 값 (잡음 제거)
            with timer() as t:
                kmeans(Xc[:n], k, iters=10, seed=0, stop=False)   # 반복 T=10 으로 고정
            best = min(best, t["s"])
        rows.append({"N": n, "k": k, "반복 T": 10, "시간(s)": best, "N·k·d·T (억)": n * k * 32 * 10 / 1e8})
cost = pd.DataFrame(rows)

nlist = 141
full = kmeans(Xb, nlist, init="pp", iters=25, seed=0)
with timer() as ts:
    Cs, _, _ = kmeans(Xb[np.random.default_rng(0).choice(N, 2_000, replace=False)], nlist, init="pp", iters=25, seed=0)
    labs = sqdist(Xb, Cs).argmin(1)                           # 전체는 배정만 (벡터당 k·d)
rec = {}
for name, (C, lab) in {"전체로 학습": (full[0], full[1]), "10% 표본으로 학습": (Cs, labs)}.items():
    L = {j: np.where(lab == j)[0] for j in range(nlist)}
    rec[name] = recall([ivf_search(qq, C, L, Xb, k=10, nprobe=4)[0] for qq in Qb], truth)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for k, g in cost.groupby("k"):
    axes[0].plot(g["N·k·d·T (억)"], g["시간(s)"], "-o", label=f"k={k}")
axes[0].set_xlabel("N·k·d·T (억 번 연산)"); axes[0].set_ylabel("학습 시간(s)"); axes[0].legend()
axes[0].set_title("학습 시간은 N·k·d·T 에 비례")
axes[1].bar(list(rec.keys()), list(rec.values()), color=["#3D484A", "#FF7043"])
for i, v in enumerate(rec.values()):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center")
axes[1].set_ylim(0, 1.05); axes[1].set_title("recall@10 (nprobe=4) — 표본 학습도 거의 같다")
plt.tight_layout(); plt.show()
cost.round(2)

**관찰:** 학습 시간은 N·k·d·T 에 거의 정비례한다. 10% 표본으로 학습한 중심도 recall 이 거의 같다.
pgvector 의 IVFFlat 도 빌드 때 표본으로 k-means 를 돌린다 (22번에서 확인).